In [1]:
%pip install --upgrade google-cloud-bigquery google-cloud-bigquery-storage pandas pyarrow db-dtypes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 922.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 42.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [2]:
import os

# Define content for setup_authentication.py
setup_auth_content = """
import os

def setup_authentication(key_path, auth_env_var):
    \"\"\"
    Sets the GOOGLE_APPLICATION_CREDENTIALS environment variable.

    Args:
        key_path (str): The path to the service account key file.
        auth_env_var (str): The name of the environment variable to set (e.g., "GOOGLE_APPLICATION_CREDENTIALS").
    \"\"\"
    os.environ[auth_env_var] = key_path
    print(f"Environment variable {auth_env_var} set to {key_path}")
"""

# Write setup_authentication.py directly in /content/
with open('setup_authentication.py', 'w') as f:
    f.write(setup_auth_content)
print("Created setup_authentication.py in /content/")

# Define content for initialize_client.py
# Correcting the internal import to reflect a peer relationship
initialize_client_content = """
from google.cloud import bigquery
from setup_authentication import setup_authentication # Corrected for peer file

def initialize_bigquery_client(project_id):
    \"\"\"
    Initializes a Google BigQuery client.

    Args:
        project_id (str): The Google Cloud project ID.

    Returns:
        google.cloud.bigquery.Client: An initialized BigQuery client.
    \"\"\"
    try:
        client = bigquery.Client(project=project_id)
        print(f"BigQuery client initialized for project: {project_id}")
        return client
    except Exception as e:
        print(f"Error initializing BigQuery client: {e}")
        return None
"""

# Write initialize_client.py directly in /content/
with open('initialize_client.py', 'w') as f:
    f.write(initialize_client_content)
print("Created initialize_client.py in /content/")

# Now, the original code from the selected cell should work
# as the necessary files are created and internal imports are corrected.
from initialize_client import initialize_bigquery_client
from setup_authentication import setup_authentication

# Définir le chemin vers le fichier de clé d'authentification BigQuery
bigquery_key_path = "/content/gdeltstef-b64ed9f7831c.json"
auth_env = "GOOGLE_APPLICATION_CREDENTIALS"

# Définir la variable d'environnement pour authentifier l'accès à Google Cloud avec la clé
setup_authentication(bigquery_key_path, auth_env)

# Initialisation le client BigQuery en utilisant la clé d'authentification configurée précédemment
project_id = "gdeltstef"
client = initialize_bigquery_client(project_id)

Created setup_authentication.py in /content/
Created initialize_client.py in /content/
Environment variable GOOGLE_APPLICATION_CREDENTIALS set to /content/gdeltstef-b64ed9f7831c.json
BigQuery client initialized for project: gdeltstef


In [3]:
from google.cloud import bigquery
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/content/gdelt_benin_2025")
OUTPUT_DIR.mkdir(exist_ok=True)

QUERY_EVENTS = """
SELECT
  GLOBALEVENTID,
  SQLDATE,
  MonthYear,
  Actor1Name,
  Actor1CountryCode,
  Actor1Type1Code,
  Actor2Name,
  Actor2CountryCode,
  Actor2Type1Code,
  IsRootEvent,
  EventCode,
  EventBaseCode,
  EventRootCode,
  QuadClass,
  GoldsteinScale,
  NumMentions,
  NumSources,
  NumArticles,
  AvgTone,
  ActionGeo_FullName,
  ActionGeo_CountryCode,
  ActionGeo_ADM1Code,
  ActionGeo_Lat,
  ActionGeo_Long,
  SOURCEURL

FROM `gdelt-bq.gdeltv2.events`

WHERE
  Year = 2025
  AND (
    ActionGeo_CountryCode = 'BN'
    OR Actor1Geo_CountryCode = 'BN'
    OR Actor2Geo_CountryCode = 'BN'
    OR Actor1CountryCode     = 'BEN'
    OR Actor2CountryCode     = 'BEN'
  )
"""

print("⏳ Exécution query Events...")
events_df = client.query(QUERY_EVENTS).to_dataframe(
    create_bqstorage_client=True,   # téléchargement rapide via Storage API
    progress_bar_type='tqdm'
)

path_events = OUTPUT_DIR / "events_benin_2025.parquet"
events_df.to_parquet(path_events, index=False, compression='snappy')

print(f"✅ Events → {len(events_df):,} lignes | {path_events.stat().st_size / 1e6:.1f} MB")
events_df.head(3)

⏳ Exécution query Events...
Query is running:   0%|          |

Forbidden: 403 Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas; reason: quotaExceeded, location: unbilled.analysis, message: Quota exceeded: Your project exceeded quota for free query bytes scanned. For more information, see https://cloud.google.com/bigquery/docs/troubleshoot-quotas

Location: US
Job ID: 47c031f5-ee78-4122-878e-bb9f4c5ed98b


In [ ]:
# Les GLOBALEVENTID filtrés sont passés via une subquery sur la table events déjà connue
# On évite de re-scanner gdeltv2.events en réutilisant les IDs en mémoire

event_ids = events_df['GLOBALEVENTID'].dropna().astype(str).unique().tolist()
print(f"🔑 {len(event_ids):,} GLOBALEVENTID à joindre")

# BigQuery accepte des listes IN jusqu'à ~10 000 valeurs sans problème
# Au-delà on passe par une temp table — géré automatiquement ci-dessous

CHUNK_SIZE = 5000  # sécurité pour la clause IN

def query_mentions_chunked(client, event_ids, chunk_size=CHUNK_SIZE):
    chunks = [event_ids[i:i+chunk_size] for i in range(0, len(event_ids), chunk_size)]
    print(f"📦 {len(chunks)} chunk(s) de {chunk_size} IDs max")
    dfs = []

    for i, chunk in enumerate(chunks):
        ids_str = ','.join(chunk)
        query = f"""
        SELECT
          GLOBALEVENTID,
          EventTimeDate,
          MentionTimeDate,
          MentionType,
          MentionSourceName,
          MentionIdentifier,
          Confidence,
          MentionDocTone

        FROM `gdelt-bq.gdeltv2.eventmentions`

        WHERE
          EventTimeDate BETWEEN 20250101000000 AND 20251231235959
          AND GLOBALEVENTID IN ({ids_str})
        """
        print(f"  ⏳ Chunk {i+1}/{len(chunks)}...")
        df = client.query(query).to_dataframe(
            create_bqstorage_client=True,
            progress_bar_type='tqdm'
        )
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True).drop_duplicates()


mentions_df = query_mentions_chunked(client, event_ids)

path_mentions = OUTPUT_DIR / "mentions_benin_2025.parquet"
mentions_df.to_parquet(path_mentions, index=False, compression='snappy')

print(f"✅ Mentions → {len(mentions_df):,} lignes | {path_mentions.stat().st_size / 1e6:.1f} MB")
mentions_df.head(3)

In [4]:
# URLs issues de Mentions → filtre GKG par URL
mention_urls = mentions_df['MentionIdentifier'].dropna().unique().tolist()
print(f"🔗 {len(mention_urls):,} URLs uniques pour jointure GKG")

def query_gkg_chunked(client, mention_urls, chunk_size=CHUNK_SIZE):
    chunks = [mention_urls[i:i+chunk_size] for i in range(0, len(mention_urls), chunk_size)]
    print(f"📦 {len(chunks)} chunk(s) · filtre URL + filtre géo V2Locations")
    dfs = []

    for i, chunk in enumerate(chunks):
        # Échapper les URLs pour SQL (apostrophes éventuelles)
        urls_str = ','.join(f"'{u.replace(chr(39), chr(39)+chr(39))}'" for u in chunk)

        query = f"""
        SELECT
          GKGRECORDID,
          DATE,
          SourceCommonName,
          DocumentIdentifier,
          V2Themes,
          V2Locations,
          V2Persons,
          V2Organizations,
          V2Tone,
          Quotations,
          AllNames,
          Amounts,
          GCAM

        FROM `gdelt-bq.gdeltv2.gkg`

        WHERE
          DATE BETWEEN 20250101000000 AND 20251231235959
          AND (
            V2Locations LIKE '%#BN#%'
            OR DocumentIdentifier IN ({urls_str})
          )
        """
        print(f"  ⏳ Chunk {i+1}/{len(chunks)}...")
        df = client.query(query).to_dataframe(
            create_bqstorage_client=True,
            progress_bar_type='tqdm'
        )
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['GKGRECORDID'])


gkg_df = query_gkg_chunked(client, mention_urls)

path_gkg = OUTPUT_DIR / "gkg_benin_2025.parquet"
gkg_df.to_parquet(path_gkg, index=False, compression='snappy')

print(f"✅ GKG → {len(gkg_df):,} lignes | {path_gkg.stat().st_size / 1e6:.1f} MB")
gkg_df.head(3)

NameError: name 'mentions_df' is not defined

In [5]:
# ── Récap final ───────────────────────────────────────────────
print("=" * 50)
print("RÉCAPITULATIF EXTRACTION GDELT · BÉNIN 2025")
print("=" * 50)
for label, df, path in [
    ("Events",   events_df,   path_events),
    ("Mentions", mentions_df, path_mentions),
    ("GKG",      gkg_df,      path_gkg),
]:
    print(f"{label:<10} {len(df):>8,} lignes | {path.stat().st_size/1e6:>6.1f} MB → {path.name}")

# ── Copie vers Google Drive (optionnel) ──────────────────────
SAVE_TO_DRIVE = True   # passer à False si non souhaité

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    drive_dir = Path("/content/drive/MyDrive/gdelt_benin_2025")
    drive_dir.mkdir(exist_ok=True)

    import shutil
    for p in [path_events, path_mentions, path_gkg]:
        shutil.copy(p, drive_dir / p.name)
        print(f"📁 Copié → Drive : {p.name}")

Query is running:   0%|          |
RÉCAPITULATIF EXTRACTION GDELT · BÉNIN 2025


NameError: name 'events_df' is not defined